In [2]:
import requests

API_KEY = "6c8ec3d465c0bcf623a06a2b4c43408c"
headers = {"api-key": API_KEY}

TARGET_ID = 738558

url = (
    f"https://www.tng-project.org/api/TNG300-1/"
    f"snapshots/99/subhalos/{TARGET_ID}/sublink/mpb.hdf5"
)

r = requests.get(url, headers=headers)

print("Status:", r.status_code)

with open("tree.hdf5", "wb") as f:
    f.write(r.content)

SSLError: HTTPSConnectionPool(host='www.tng-project.org', port=443): Max retries exceeded with url: /api/TNG300-1/snapshots/99/subhalos/738558/sublink/mpb.hdf5 (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1081)')))

In [3]:
import h5py
import numpy as np

with h5py.File("tree.hdf5", "r") as f:
    print(list(f.keys()))

['DescendantID', 'FirstProgenitorID', 'FirstSubhaloInFOFGroupID', 'GroupBHMass', 'GroupBHMdot', 'GroupCM', 'GroupFirstSub', 'GroupGasMetalFractions', 'GroupGasMetallicity', 'GroupLen', 'GroupLenType', 'GroupMass', 'GroupMassType', 'GroupNsubs', 'GroupPos', 'GroupSFR', 'GroupStarMetalFractions', 'GroupStarMetallicity', 'GroupVel', 'GroupWindMass', 'Group_M_Crit200', 'Group_M_Crit500', 'Group_M_Mean200', 'Group_M_TopHat200', 'Group_R_Crit200', 'Group_R_Crit500', 'Group_R_Mean200', 'Group_R_TopHat200', 'LastProgenitorID', 'MainLeafProgenitorID', 'Mass', 'MassHistory', 'NextProgenitorID', 'NextSubhaloInFOFGroupID', 'NumParticles', 'RootDescendantID', 'SnapNum', 'SubfindID', 'SubhaloBHMass', 'SubhaloBHMdot', 'SubhaloBfldDisk', 'SubhaloBfldHalo', 'SubhaloCM', 'SubhaloGasMetalFractions', 'SubhaloGasMetalFractionsHalfRad', 'SubhaloGasMetalFractionsMaxRad', 'SubhaloGasMetalFractionsSfr', 'SubhaloGasMetalFractionsSfrWeighted', 'SubhaloGasMetallicity', 'SubhaloGasMetallicityHalfRad', 'SubhaloGasM

In [4]:
with h5py.File("tree.hdf5", "r") as f:

    snap = f["SnapNum"][:]
    subfind = f["SubfindID"][:]

    matches = np.where(snap == 66)[0]

    print("Number of snapshot-66 entries:", len(matches))

    for i in matches:
        print(
            "Snap:", snap[i],
            "SubfindID:", subfind[i]
        )

Number of snapshot-66 entries: 1
Snap: 66 SubfindID: 382115


In [5]:
SNAP = 66
SUBFIND_ID = 382115

url = (
    f"https://www.tng-project.org/api/TNG300-1/"
    f"snapshots/{SNAP}/subhalos/{SUBFIND_ID}/info.json"
)

r = requests.get(url, headers=headers)

print("Status:", r.status_code)

d = r.json()

print("Position:", d["SubhaloPos"])
print("Group:", d["SubhaloGrNr"])
print("Parent:", d.get("SubhaloParent"))
print("Mass:", d["SubhaloMass"])
print("HalfmassRad:", d["SubhaloHalfmassRad"])

Status: 200
Position: [44581.32421875, 162441.15625, 47623.71875]
Group: 716
Parent: 0
Mass: 25.586179733276367
HalfmassRad: 16.989349365234375


In [6]:
GROUP_ID = d["SubhaloGrNr"]

url = (
    f"https://www.tng-project.org/api/TNG300-1/"
    f"snapshots/{SNAP}/halos/{GROUP_ID}/info.json"
)

r = requests.get(url, headers=headers)

print("Status:", r.status_code)

group = r.json()

print("Group position:", group["GroupPos"])
print("R200:", group["Group_R_Crit200"])
print("M200:", group["Group_M_Crit200"])

Status: 200
Group position: [44717.41015625, 162461.765625, 47668.10546875]
R200: 619.4302368164062
M200: 2783.8896484375


In [7]:
import numpy as np

galaxy_pos = np.array(d["SubhaloPos"])
group_pos = np.array(group["GroupPos"])

BOX_SIZE = 205000.0 
delta = galaxy_pos - group_pos
delta = delta - BOX_SIZE * np.round(delta / BOX_SIZE)

distance = np.linalg.norm(delta)

R200 = group["Group_R_Crit200"]

r_over_R200 = distance / R200

print("Galaxy-group distance:", distance, "ckpc/h")
print("R200:", R200, "ckpc/h")
print("r/R200:", r_over_R200)

Galaxy-group distance: 144.6178050052487 ckpc/h
R200: 619.4302368164062 ckpc/h
r/R200: 0.2334690759503756


In [8]:
url = (
    f"https://www.tng-project.org/api/TNG300-1/"
    f"snapshots/66/subhalos/{SUBFIND_ID}/sublink/simple.json"
)

r = requests.get(url, headers=headers)

print("Status:", r.status_code)

data = r.json()

print("\nMAIN PROGENITOR:")
for item in data.get("Main", []):
    print(item)

print("\nMERGERS:")
for item in data.get("Mergers", []):
    print(item)

Status: 200

MAIN PROGENITOR:
[99, 738558]
[98, 734385]
[97, 721535]
[96, 721142]
[95, 709726]
[94, 704856]
[93, 701739]
[92, 694059]
[91, 682178]
[90, 674042]
[89, 663605]
[88, 650474]
[87, 642893]
[86, 632050]
[85, 620627]
[84, 610044]
[83, 594290]
[82, 583768]
[81, 569325]
[80, 555423]
[79, 547182]
[78, 528267]
[77, 518174]
[76, 500795]
[75, 485994]
[74, 476958]
[73, 464255]
[72, 448572]
[71, 434963]
[70, 425167]
[69, 413531]
[68, 403899]
[67, 393607]
[66, 382115]
[65, 374065]
[64, 364735]
[63, 357250]
[62, 349626]
[61, 362352]
[60, 342637]
[59, 338791]
[58, 330725]
[57, 323259]
[56, 311030]
[55, 317979]
[54, 304444]
[53, 295530]
[52, 288280]
[51, 282304]
[50, 355694]
[49, 402296]
[48, 393872]
[47, 378031]
[46, 372236]
[45, 343996]
[44, 960935]
[43, 934613]
[42, 887568]
[41, 842182]
[40, 792094]
[39, 770783]
[38, 727807]
[37, 691473]
[36, 762029]
[35, 741496]
[34, 716508]
[33, 685533]
[32, 662855]
[31, 688911]
[30, 824921]
[29, 835880]
[28, 836004]
[27, 776220]
[26, 1101203]
[25, 10

In [9]:
import requests
import numpy as np
import pandas as pd

API_KEY = "6c8ec3d465c0bcf623a06a2b4c43408c"
headers = {"api-key": API_KEY}

Z0_ID = 382115
BOX_SIZE = 205000.0 
url = (
    f"https://www.tng-project.org/api/TNG300-1/"
    f"snapshots/99/subhalos/{Z0_ID}/sublink/simple.json"
)

r = requests.get(url, headers=headers)
print("Tree status:", r.status_code)

tree = r.json()
main = {
    snap: subfind
    for snap, subfind in tree["Main"]
    if 60 <= snap <= 72
}

print("\nMain branch:")
for snap in sorted(main):
    print(f"Snap {snap}: SubfindID {main[snap]}")

results = []

for snap in sorted(main):

    subfind = main[snap]
    sub_url = (
        f"https://www.tng-project.org/api/TNG300-1/"
        f"snapshots/{snap}/subhalos/{subfind}/info.json"
    )

    sub_r = requests.get(sub_url, headers=headers)

    if sub_r.status_code != 200:
        print(f"ERROR: snap {snap}, subhalo {subfind}")
        continue

    sub = sub_r.json()

    galaxy_pos = np.array(sub["SubhaloPos"])
    group_id = sub["SubhaloGrNr"]
    group_url = (
        f"https://www.tng-project.org/api/TNG300-1/"
        f"snapshots/{snap}/halos/{group_id}/info.json"
    )

    group_r = requests.get(group_url, headers=headers)

    if group_r.status_code != 200:
        print(f"ERROR: snap {snap}, group {group_id}")
        continue

    group = group_r.json()

    group_pos = np.array(group["GroupPos"])
    R200 = group["Group_R_Crit200"]

    delta = galaxy_pos - group_pos
    delta -= BOX_SIZE * np.round(delta / BOX_SIZE)

    distance = np.linalg.norm(delta)
    r_over_R200 = distance / R200

    results.append({
        "Snapshot": snap,
        "SubfindID": subfind,
        "GroupID": group_id,

        "GalaxyX": galaxy_pos[0],
        "GalaxyY": galaxy_pos[1],
        "GalaxyZ": galaxy_pos[2],

        "GroupX": group_pos[0],
        "GroupY": group_pos[1],
        "GroupZ": group_pos[2],

        "Distance_ckpch": distance,
        "R200_ckpch": R200,
        "r_over_R200": r_over_R200,

        "SubhaloMass": sub["SubhaloMass"],
        "HalfmassRad": sub["SubhaloHalfmassRad"]
    })

df = pd.DataFrame(results)

print("\n========================================")
print("HALO 382115: HOST-CENTER TRAJECTORY")
print("========================================")

print(
    df[
        [
            "Snapshot",
            "SubfindID",
            "GroupID",
            "Distance_ckpch",
            "R200_ckpch",
            "r_over_R200"
        ]
    ].to_string(index=False)
)

Tree status: 200

Main branch:
Snap 60: SubfindID 10658557
Snap 61: SubfindID 12835907
Snap 62: SubfindID 11205356
Snap 63: SubfindID 10420124
Snap 64: SubfindID 10283986
Snap 65: SubfindID 10944346
Snap 66: SubfindID 10285871
Snap 67: SubfindID 9974996
Snap 68: SubfindID 9851800
Snap 69: SubfindID 9735214
Snap 70: SubfindID 10224827
Snap 71: SubfindID 10590266
Snap 72: SubfindID 11128409

HALO 382115: HOST-CENTER TRAJECTORY
 Snapshot  SubfindID  GroupID  Distance_ckpch  R200_ckpch  r_over_R200
       60   10658557  8541976             0.0   24.772589          0.0
       61   12835907 11890323             0.0   24.205507          0.0
       62   11205356  9364414             0.0   24.270277          0.0
       63   10420124  8281933             0.0   24.148382          0.0
       64   10283986  8118995             0.0   23.844973          0.0
       65   10944346  9074428             0.0   23.468109          0.0
       66   10285871  8164573             0.0   23.127796          0.0
   

In [10]:
import requests
import numpy as np
import pandas as pd

API_KEY = "6c8ec3d465c0bcf623a06a2b4c43408c"
headers = {"api-key": API_KEY}

Z0_ID = 382115

# Get main progenitor branch
url = (
    f"https://www.tng-project.org/api/TNG300-1/"
    f"snapshots/99/subhalos/{Z0_ID}/sublink/simple.json"
)

tree = requests.get(url, headers=headers).json()

main = {
    snap: subfind
    for snap, subfind in tree["Main"]
    if 60 <= snap <= 72
}

results = []

for snap in sorted(main):

    subfind = main[snap]

    # Subhalo
    sub_url = (
        f"https://www.tng-project.org/api/TNG300-1/"
        f"snapshots/{snap}/subhalos/{subfind}/info.json"
    )

    sub = requests.get(sub_url, headers=headers).json()

    galaxy_pos = np.array(sub["SubhaloPos"])
    group_id = sub["SubhaloGrNr"]

    # Host FoF group
    group_url = (
        f"https://www.tng-project.org/api/TNG300-1/"
        f"snapshots/{snap}/halos/{group_id}/info.json"
    )

    group = requests.get(group_url, headers=headers).json()

    group_pos = np.array(group["GroupPos"])
    group_first_sub = group["GroupFirstSub"]

    # Distance
    delta = galaxy_pos - group_pos

    # Periodic boundary correction
    BOX_SIZE = 205000.0
    delta -= BOX_SIZE * np.round(delta / BOX_SIZE)

    distance = np.linalg.norm(delta)

    results.append({
        "snap": snap,
        "subfind": subfind,
        "group": group_id,
        "GroupFirstSub": group_first_sub,
        "is_central": subfind == group_first_sub,
        "galaxy_pos": galaxy_pos,
        "group_pos": group_pos,
        "dx": delta[0],
        "dy": delta[1],
        "dz": delta[2],
        "distance": distance,
        "R200": group["Group_R_Crit200"],
        "r_R200": distance / group["Group_R_Crit200"]
    })

df = pd.DataFrame(results)

print(df[
    [
        "snap",
        "subfind",
        "group",
        "GroupFirstSub",
        "is_central",
        "distance",
        "R200",
        "r_R200"
    ]
].to_string(index=False))

 snap  subfind    group  GroupFirstSub  is_central  distance      R200  r_R200
   60 10658557  8541976       10658557        True       0.0 24.772589     0.0
   61 12835907 11890323       12835907        True       0.0 24.205507     0.0
   62 11205356  9364414       11205356        True       0.0 24.270277     0.0
   63 10420124  8281933       10420124        True       0.0 24.148382     0.0
   64 10283986  8118995       10283986        True       0.0 23.844973     0.0
   65 10944346  9074428       10944346        True       0.0 23.468109     0.0
   66 10285871  8164573       10285871        True       0.0 23.127796     0.0
   67  9974996  7759251        9974996        True       0.0 23.192663     0.0
   68  9851800  7612099        9851800        True       0.0 22.850281     0.0
   69  9735214  7474262        9735214        True       0.0 22.502581     0.0
   70 10224827  8158693       10224827        True       0.0 22.574181     0.0
   71 10590266  8699725       10590266        True  

In [13]:
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

# ============================================================
# MERGER MASS RATIOS
# ============================================================

# Your TNG API key
API_KEY = "6c8ec3d465c0bcf623a06a2b4c43408c"

headers = {"api-key": API_KEY}
print(tree)
primary_lookup = {}

for snap, subfind_id, mass in zip(
    tree["SnapNum"][:],
    tree["SubfindID"][:],
    tree["Mass"][:]
):
    primary_lookup[int(snap)] = {
        "SubfindID": int(subfind_id),
        "Mass": float(mass)
    }

print(f"Primary snapshots available: {len(primary_lookup)}")

# ------------------------------------------------------------
# 2. Function to retrieve a SECONDARY galaxy's mass
# ------------------------------------------------------------
# ============================================================
# GET SECONDARY MASSES AND CALCULATE MERGER RATIOS
# ============================================================

import requests
import pandas as pd

API_KEY = "YOUR_API_KEY_HERE"

headers = {
    "api-key": API_KEY
}


def get_subhalo_mass(snap, subfind_id):

    url = (
        f"https://www.tng-project.org/api/TNG300-1/"
        f"snapshots/{snap}/subhalos/{subfind_id}/"
    )

    response = requests.get(
        url,
        headers=headers
    )

    if response.status_code != 200:
        print(
            f"API error for snapshot {snap}, "
            f"subhalo {subfind_id}: "
            f"{response.status_code}"
        )
        return None

    data = response.json()

    return data.get("mass")


results = []

for snap, secondary_id in mergers:

    snap = int(snap)
    secondary_id = int(secondary_id)

    # Find the primary/main-branch galaxy
    if snap not in primary_lookup:

        print(
            f"Could not find primary at snapshot {snap}"
        )
        continue

    primary_id = primary_lookup[snap]["SubfindID"]
    primary_mass = primary_lookup[snap]["Mass"]

    # Get secondary/merging galaxy mass
    secondary_mass = get_subhalo_mass(
        snap,
        secondary_id
    )

    if secondary_mass is None:

        print(
            f"Could not find mass for merger "
            f"{snap}, {secondary_id}"
        )
        continue

    # Mass ratio
    mass_ratio = secondary_mass / primary_mass

    # Classification
    if mass_ratio >= 0.25:
        merger_type = "Major"
    elif mass_ratio >= 0.10:
        merger_type = "Minor"
    else:
        merger_type = "Very minor"

    results.append({
        "Snapshot": snap,
        "PrimaryID": primary_id,
        "SecondaryID": secondary_id,
        "PrimaryMass": primary_mass,
        "SecondaryMass": secondary_mass,
        "MassRatio": mass_ratio,
        "MergerType": merger_type
    })


# Create dataframe
merger_df = pd.DataFrame(results)

if len(merger_df) > 0:

    merger_df = merger_df.sort_values(
        "Snapshot"
    ).reset_index(drop=True)

    print("=" * 100)
    print("MERGER MASS RATIOS")
    print("=" * 100)

    display(merger_df)

else:

    print("No merger mass ratios were successfully calculated.")

{'Main': [[99, 382115], [98, -1], [97, 3500671], [96, 3303069], [95, 3875894], [94, 4501145], [93, 7549653], [92, 7810149], [91, 8315861], [90, 8744897], [89, 9109936], [88, 9287804], [87, 9720754], [86, 9747927], [85, 10254225], [84, 10391379], [83, 9653860], [82, 11446201], [81, 11340669], [80, 9731424], [79, 9422900], [78, 8835382], [77, 8982547], [76, 9360850], [75, 9707853], [74, 9567173], [73, 10518647], [72, 11128409], [71, 10590266], [70, 10224827], [69, 9735214], [68, 9851800], [67, 9974996], [66, 10285871], [65, 10944346], [64, 10283986], [63, 10420124], [62, 11205356], [61, 12835907], [60, 10658557], [59, 10070432], [58, 10569993], [57, 10429831], [56, 10065485], [55, 10445478], [54, 10172220], [53, 10572083], [52, 11555342], [51, 11493153], [50, 11938141], [49, 12142130], [48, 12215847], [47, 12161468], [46, 13245396], [45, 13030189], [44, 12536925], [43, 12373993], [42, 12108094], [41, 12696613], [40, 13548707], [39, 13306865], [38, 12877493], [37, 15916405], [36, 16038368

KeyError: 'SnapNum'

In [16]:
print(type(tree))
print(tree)
print(tree.keys() if hasattr(tree, "keys") else "NO .keys()")

<class 'dict'>
{'Main': [[99, 382115], [98, -1], [97, 3500671], [96, 3303069], [95, 3875894], [94, 4501145], [93, 7549653], [92, 7810149], [91, 8315861], [90, 8744897], [89, 9109936], [88, 9287804], [87, 9720754], [86, 9747927], [85, 10254225], [84, 10391379], [83, 9653860], [82, 11446201], [81, 11340669], [80, 9731424], [79, 9422900], [78, 8835382], [77, 8982547], [76, 9360850], [75, 9707853], [74, 9567173], [73, 10518647], [72, 11128409], [71, 10590266], [70, 10224827], [69, 9735214], [68, 9851800], [67, 9974996], [66, 10285871], [65, 10944346], [64, 10283986], [63, 10420124], [62, 11205356], [61, 12835907], [60, 10658557], [59, 10070432], [58, 10569993], [57, 10429831], [56, 10065485], [55, 10445478], [54, 10172220], [53, 10572083], [52, 11555342], [51, 11493153], [50, 11938141], [49, 12142130], [48, 12215847], [47, 12161468], [46, 13245396], [45, 13030189], [44, 12536925], [43, 12373993], [42, 12108094], [41, 12696613], [40, 13548707], [39, 13306865], [38, 12877493], [37, 15916405]